# 12 — Score Threshold Experiment

Test whether shifting the rounding threshold from λ=1.0 to λ=0.85 improves
exact score accuracy on the **held-out** 2022 WC fold.

**Setup (same as notebook 07 CV strategy):**
- Train ensemble (LightGBM 90% + XGBoost 10%) on all non-2022-WC data
- Both models use symmetric mirroring internally (built into LGBMGoalModel/XGBGoalModel)
- Test on the 64 held-out 2022 WC matches

**Two scoring methods compared:**
- `mode`: Poisson grid argmax — switches 0→1 at λ=1.0 (current production)
- `th085`: floor(λ + 0.15) — switches 0→1 at λ=0.85, 1→2 at λ=1.85, etc.

No existing files are modified.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from src.models.base import load_model_dataset
from src.features.feature_columns import FEATURE_COLS
from src.models.weighting import apply_competition_weights
from src.models.lgbm_model import LGBMGoalModel
from src.models.xgb_model import XGBGoalModel
from src.models.ensemble import EnsembleGoalModel
from src.models.score_conversion import win_draw_loss_probs, most_likely_score
from src.evaluation.metrics import rps_batch

TARGET_COLS = ['goals_A', 'goals_B']
print('Imports OK')

Imports OK


## 1. Load data and create the 2022 CV fold

In [2]:
df = load_model_dataset('../data/processed/updated_model_dataset.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
print(f'Dataset: {len(df)} matches, {df["date"].min().date()} to {df["date"].max().date()}')

# Val = 2022 WC matches only
val_mask = (
    (df['tournament_year'] == 2022) &
    (df['competition'].str.strip().str.lower() == 'world cup')
)
wc_start = df[val_mask]['date'].min()
print(f'2022 WC start date: {wc_start.date()}')

# Train = everything strictly before the WC start (proper temporal split)
train_mask = df['date'] < wc_start

train_df = df[train_mask]
val_df   = df[val_mask]

X_train = train_df[FEATURE_COLS].fillna(0)
y_train = train_df[TARGET_COLS].values
X_val   = val_df[FEATURE_COLS].fillna(0)
y_val   = val_df[TARGET_COLS].values

w_train = apply_competition_weights(train_df)
meta    = val_df[['date', 'team_A', 'team_B']].reset_index(drop=True)

print(f'Train: {len(X_train)} matches (before {wc_start.date()})  |  Val (2022 WC): {len(X_val)} matches')

Dataset: 21655 matches, 2004-01-01 to 2026-06-08
2022 WC start date: 2022-11-20
Train: 17844 matches (before 2022-11-20)  |  Val (2022 WC): 64 matches


## 2. Train ensemble on non-2022 data

Uses the same tuned hyperparameters as notebook 07 (baked into the model defaults).
Symmetric mirroring is built into `LGBMGoalModel` and `XGBGoalModel` — no extra steps needed.

In [19]:
lgbm = LGBMGoalModel()   # default params from Optuna CV in notebook 07
xgb  = XGBGoalModel()
ensemble = EnsembleGoalModel([lgbm, xgb], weights=[0.8, 0.2])

print('Training ensemble on non-2022-WC data...')
ensemble.fit(X_train, y_train, sample_weight=w_train)
print('Done.')

lambdas = np.clip(ensemble.predict(X_val), 0, None)   # shape (64, 2)
print(f'Lambda range — A: [{lambdas[:,0].min():.2f}, {lambdas[:,0].max():.2f}]  '
      f'B: [{lambdas[:,1].min():.2f}, {lambdas[:,1].max():.2f}]')
print(f'Mean lambda   — A: {lambdas[:,0].mean():.3f}  B: {lambdas[:,1].mean():.3f}')

Training ensemble on non-2022-WC data...
Done.
Lambda range — A: [0.29, 3.29]  B: [0.24, 3.32]
Mean lambda   — A: 1.797  B: 0.754


## 3. Two scoring functions

In [20]:
def score_mode(la, lb):
    """Poisson grid argmax — switches 0→1 at λ=1.0 (current production)."""
    return most_likely_score(la, lb)


def score_th(la, lb, threshold=0.9):
    """Shifted floor — switches 0→1 at λ=threshold, 1→2 at λ=1+threshold, etc."""
    shift = 1.0 - threshold    # 0.15 for threshold=0.85
    return int(la + shift), int(lb + shift)


# Show where they differ
print(f'{"λ":>6}  {"mode":>6}  {"th=0.85":>8}')
print('-' * 25)
for lam in [0.50, 0.70, 0.84, 0.85, 0.90, 0.99, 1.00, 1.20, 1.84, 1.85, 1.90, 2.00, 2.50]:
    pm = score_mode(lam, 0.0)[0]
    pt = score_th(lam, 0.0)[0]
    flag = ' ←' if pm != pt else ''
    print(f'{lam:>6.2f}  {pm:>6}  {pt:>8}{flag}')

     λ    mode   th=0.85
-------------------------
  0.50       0         0
  0.70       0         0
  0.84       0         0
  0.85       0         0
  0.90       0         1 ←
  0.99       0         1 ←
  1.00       0         1 ←
  1.20       1         1
  1.84       1         1
  1.85       1         1
  1.90       1         2 ←
  2.00       1         2 ←
  2.50       2         2


## 4. Evaluate both methods on 2022 WC

In [21]:
def evaluate(y_true, lambdas, score_fn, label):
    rows = []
    for i in range(len(y_true)):
        la, lb = float(lambdas[i, 0]), float(lambdas[i, 1])
        pa, pb = score_fn(la, lb)
        ta, tb = int(y_true[i, 0]), int(y_true[i, 1])
        true_result = 'W' if ta > tb else ('D' if ta == tb else 'L')
        pred_result = 'W' if pa > pb else ('D' if pa == pb else 'L')
        rows.append({
            'team_A':    meta.iloc[i]['team_A'],
            'team_B':    meta.iloc[i]['team_B'],
            'actual':    f'{ta}-{tb}',
            'predicted': f'{pa}-{pb}',
            'lambda':    f'{la:.2f}-{lb:.2f}',
            'exact':     (pa == ta) and (pb == tb),
            'result_ok': true_result == pred_result,
        })
    results = pd.DataFrame(rows)
    n = len(results)
    n_exact  = results['exact'].sum()
    n_result = results['result_ok'].sum()
    probs = np.array([win_draw_loss_probs(la, lb) for la, lb in lambdas])
    rps   = rps_batch(y_true, probs)
    print(f'\n=== {label} ===')
    print(f'  Exact score:   {n_exact}/{n}  ({n_exact/n*100:.1f}%)')
    print(f'  Result W/D/L:  {n_result}/{n}  ({n_result/n*100:.1f}%)')
    print(f'  RPS:           {rps:.4f}  (lower is better; unaffected by scoring method)')
    return results


res_mode = evaluate(y_val, lambdas, score_mode, 'Poisson mode — current production')
res_th85 = evaluate(y_val, lambdas, lambda la, lb: score_th(la, lb, 0.9), 'Threshold 0.9')


=== Poisson mode — current production ===
  Exact score:   19/64  (29.7%)
  Result W/D/L:  50/64  (78.1%)
  RPS:           0.0716  (lower is better; unaffected by scoring method)

=== Threshold 0.9 ===
  Exact score:   20/64  (31.2%)
  Result W/D/L:  52/64  (81.2%)
  RPS:           0.0716  (lower is better; unaffected by scoring method)


## 5. Which matches changed prediction?

In [22]:
changed = res_mode['predicted'] != res_th85['predicted']
print(f'{changed.sum()} of 64 matches changed prediction:\n')

diff = pd.DataFrame({
    'team_A':     meta['team_A'],
    'team_B':     meta['team_B'],
    'actual':     res_mode['actual'],
    'lambda':     res_mode['lambda'],
    'pred_mode':  res_mode['predicted'],
    'pred_th85':  res_th85['predicted'],
    'exact_mode': res_mode['exact'],
    'exact_th85': res_th85['exact'],
})[changed].reset_index(drop=True)

diff['verdict'] = diff.apply(
    lambda r: '✓ threshold wins' if r['exact_th85'] and not r['exact_mode']
              else ('✓ mode wins'    if r['exact_mode'] and not r['exact_th85']
              else 'neither'),
    axis=1
)
print(diff.to_string(index=False))

8 of 64 matches changed prediction:

   team_A       team_B actual    lambda pred_mode pred_th85  exact_mode  exact_th85          verdict
 Cameroon       Serbia    3-3 0.96-1.13       0-1       1-1       False       False          neither
  England        Wales    3-0 2.97-0.58       2-0       3-0       False        True ✓ threshold wins
  Tunisia       France    1-0 0.85-0.96       0-0       0-1       False       False          neither
   Mexico Saudi Arabia    2-1 1.94-0.40       1-0       2-0       False       False          neither
  Morocco       Canada    2-1 1.93-0.71       1-0       2-0       False       False          neither
  Croatia        Japan    1-1 0.92-1.07       0-1       1-1       False        True ✓ threshold wins
  Morocco     Portugal    1-0 1.51-0.95       1-0       1-1        True       False      ✓ mode wins
Argentina       France    3-3 0.90-1.19       0-1       1-1       False       False          neither


## 6. Lambda distribution — how many lambdas are in the affected zones?

In [23]:
all_lambdas = lambdas.flatten()   # 128 values (2 teams × 64 matches)
print('Lambda distribution across all 128 team-match slots:')
bins = [
    (0.00, 0.9, '  mode=0, th85=0'),
    (0.9, 1.00, '  mode=0, th85=1  ← DIFFERS'),
    (1.00, 1.9, '  mode=1, th85=1'),
    (1.9, 2.00, '  mode=1, th85=2  ← DIFFERS'),
    (2.00, 2.9, '  mode=2, th85=2'),
    (2.9, 3.00, '  mode=2, th85=3  ← DIFFERS'),
    (3.00, 10.0, '  mode=3+, th85=3+'),
]
for lo, hi, note in bins:
    n = ((all_lambdas >= lo) & (all_lambdas < hi)).sum()
    bar = '█' * n
    print(f'  [{lo:.2f}, {hi:.2f}): {n:3d}  {bar}{note}')

Lambda distribution across all 128 team-match slots:
  [0.00, 0.90):  62  ██████████████████████████████████████████████████████████████  mode=0, th85=0
  [0.90, 1.00):   5  █████  mode=0, th85=1  ← DIFFERS
  [1.00, 1.90):  23  ███████████████████████  mode=1, th85=1
  [1.90, 2.00):   2  ██  mode=1, th85=2  ← DIFFERS
  [2.00, 2.90):  30  ██████████████████████████████  mode=2, th85=2
  [2.90, 3.00):   1  █  mode=2, th85=3  ← DIFFERS
  [3.00, 10.00):   5  █████  mode=3+, th85=3+


## 7. Summary

In [12]:
n = len(y_val)
exact_mode = res_mode['exact'].sum()
exact_th85 = res_th85['exact'].sum()
gained = ((~res_mode['exact']) & res_th85['exact']).sum()
lost   = (res_mode['exact'] & (~res_th85['exact'])).sum()

print('=' * 50)
print('SUMMARY — 2022 WC held-out fold (64 matches)')
print('=' * 50)
print(f'  Poisson mode (production):  {exact_mode}/{n}  ({exact_mode/n*100:.1f}%)')
print(f'  Threshold 0.85:             {exact_th85}/{n}  ({exact_th85/n*100:.1f}%)')
print(f'  Delta: {exact_th85 - exact_mode:+d} matches ({(exact_th85-exact_mode)/n*100:+.1f}%)')
print(f'  Gained: {gained} correct  |  Lost: {lost} correct')
print()
print('Note: RPS is identical for both methods — it uses the lambda')
print('directly (Poisson probs), not the rounded score.')

SUMMARY — 2022 WC held-out fold (64 matches)
  Poisson mode (production):  18/64  (28.1%)
  Threshold 0.85:             20/64  (31.2%)
  Delta: +2 matches (+3.1%)
  Gained: 3 correct  |  Lost: 1 correct

Note: RPS is identical for both methods — it uses the lambda
directly (Poisson probs), not the rounded score.


---\n## 2018 WC — Same experiment, threshold 0.90\n\nTrain on all data before the 2018 WC start date, test on the 64 held-out 2018 WC matches.\nCompares Poisson mode vs threshold 0.90 (switches 0→1 at λ=0.90, 1→2 at λ=1.90, etc.).

In [16]:
# --- 2018 WC fold ---
val_mask_18 = (
    (df['tournament_year'] == 2018) &
    (df['competition'].str.strip().str.lower() == 'world cup')
)
wc_start_18 = df[val_mask_18]['date'].min()
print(f'2018 WC start date: {wc_start_18.date()}')

train_df_18 = df[df['date'] < wc_start_18]
val_df_18   = df[val_mask_18]

X_train_18 = train_df_18[FEATURE_COLS].fillna(0)
y_train_18 = train_df_18[TARGET_COLS].values
X_val_18   = val_df_18[FEATURE_COLS].fillna(0)
y_val_18   = val_df_18[TARGET_COLS].values
w_train_18 = apply_competition_weights(train_df_18)
meta_18    = val_df_18[['date', 'team_A', 'team_B']].reset_index(drop=True)

print(f'Train: {len(X_train_18)} matches (before {wc_start_18.date()})  |  Val (2018 WC): {len(X_val_18)} matches')

# Train fresh ensemble
lgbm_18 = LGBMGoalModel()
xgb_18  = XGBGoalModel()
ensemble_18 = EnsembleGoalModel([lgbm_18, xgb_18], weights=[0.8, 0.2])

print('Training ensemble...')
ensemble_18.fit(X_train_18, y_train_18, sample_weight=w_train_18)
print('Done.')

lambdas_18 = np.clip(ensemble_18.predict(X_val_18), 0, None)
print(f'Lambda range — A: [{lambdas_18[:,0].min():.2f}, {lambdas_18[:,0].max():.2f}]  '
      f'B: [{lambdas_18[:,1].min():.2f}, {lambdas_18[:,1].max():.2f}]')
print(f'Mean lambda   — A: {lambdas_18[:,0].mean():.3f}  B: {lambdas_18[:,1].mean():.3f}')

2018 WC start date: 2018-06-14
Train: 13713 matches (before 2018-06-14)  |  Val (2018 WC): 64 matches
Training ensemble...
Done.
Lambda range — A: [0.36, 3.66]  B: [0.35, 3.06]
Mean lambda   — A: 1.912  B: 0.699


In [17]:
def evaluate_18(y_true, lambdas, score_fn, label, meta):
    rows = []
    for i in range(len(y_true)):
        la, lb = float(lambdas[i, 0]), float(lambdas[i, 1])
        pa, pb = score_fn(la, lb)
        ta, tb = int(y_true[i, 0]), int(y_true[i, 1])
        true_result = 'W' if ta > tb else ('D' if ta == tb else 'L')
        pred_result = 'W' if pa > pb else ('D' if pa == pb else 'L')
        rows.append({
            'team_A':    meta.iloc[i]['team_A'],
            'team_B':    meta.iloc[i]['team_B'],
            'actual':    f'{ta}-{tb}',
            'predicted': f'{pa}-{pb}',
            'lambda':    f'{la:.2f}-{lb:.2f}',
            'exact':     (pa == ta) and (pb == tb),
            'result_ok': true_result == pred_result,
        })
    results = pd.DataFrame(rows)
    n = len(results)
    n_exact  = results['exact'].sum()
    n_result = results['result_ok'].sum()
    probs = np.array([win_draw_loss_probs(la, lb) for la, lb in lambdas])
    rps   = rps_batch(y_true, probs)
    print(f'\n=== {label} ===')
    print(f'  Exact score:   {n_exact}/{n}  ({n_exact/n*100:.1f}%)')
    print(f'  Result W/D/L:  {n_result}/{n}  ({n_result/n*100:.1f}%)')
    print(f'  RPS:           {rps:.4f}')
    return results


res18_mode = evaluate_18(y_val_18, lambdas_18, score_mode,
                         'Poisson mode — 2018 WC', meta_18)
res18_th90 = evaluate_18(y_val_18, lambdas_18,
                         lambda la, lb: score_th(la, lb, 0.90),
                         'Threshold 0.90 — 2018 WC', meta_18)


=== Poisson mode — 2018 WC ===
  Exact score:   17/64  (26.6%)
  Result W/D/L:  53/64  (82.8%)
  RPS:           0.0638

=== Threshold 0.90 — 2018 WC ===
  Exact score:   14/64  (21.9%)
  Result W/D/L:  52/64  (81.2%)
  RPS:           0.0638


In [18]:
# Changed predictions
changed_18 = res18_mode['predicted'] != res18_th90['predicted']
print(f'{changed_18.sum()} of 64 matches changed prediction:\n')

diff_18 = pd.DataFrame({
    'team_A':     meta_18['team_A'],
    'team_B':     meta_18['team_B'],
    'actual':     res18_mode['actual'],
    'lambda':     res18_mode['lambda'],
    'pred_mode':  res18_mode['predicted'],
    'pred_th90':  res18_th90['predicted'],
    'exact_mode': res18_mode['exact'],
    'exact_th90': res18_th90['exact'],
})[changed_18].reset_index(drop=True)

diff_18['verdict'] = diff_18.apply(
    lambda r: '✓ threshold wins' if r['exact_th90'] and not r['exact_mode']
              else ('✓ mode wins'    if r['exact_mode'] and not r['exact_th90']
              else 'neither'),
    axis=1
)
print(diff_18.to_string(index=False))

# Lambda distribution for 2018
print('\nLambda distribution (2018 WC, 128 slots):')
all_l18 = lambdas_18.flatten()
bins_18 = [
    (0.00, 0.90, '  mode=0, th90=0'),
    (0.90, 1.00, '  mode=0, th90=1  ← DIFFERS'),
    (1.00, 1.90, '  mode=1, th90=1'),
    (1.90, 2.00, '  mode=1, th90=2  ← DIFFERS'),
    (2.00, 2.90, '  mode=2, th90=2'),
    (2.90, 3.00, '  mode=2, th90=3  ← DIFFERS'),
    (3.00, 10.0, '  mode=3+, th90=3+'),
]
for lo, hi, note in bins_18:
    n = ((all_l18 >= lo) & (all_l18 < hi)).sum()
    print(f'  [{lo:.2f}, {hi:.2f}): {n:3d}  {"█" * n}{note}')

# 2018 summary
n18 = len(y_val_18)
e_mode = res18_mode['exact'].sum()
e_th90 = res18_th90['exact'].sum()
gained = ((~res18_mode['exact']) & res18_th90['exact']).sum()
lost   = (res18_mode['exact'] & (~res18_th90['exact'])).sum()
print(f'\n{"="*50}')
print(f'SUMMARY — 2018 WC held-out fold (64 matches)')
print(f'{"="*50}')
print(f'  Poisson mode:     {e_mode}/{n18}  ({e_mode/n18*100:.1f}%)')
print(f'  Threshold 0.90:   {e_th90}/{n18}  ({e_th90/n18*100:.1f}%)')
print(f'  Delta: {e_th90 - e_mode:+d} matches ({(e_th90-e_mode)/n18*100:+.1f}%)')
print(f'  Gained: {gained} correct  |  Lost: {lost} correct')

7 of 64 matches changed prediction:

   team_A      team_B actual    lambda pred_mode pred_th90  exact_mode  exact_th90     verdict
 Portugal       Spain    3-3 0.80-0.95       0-0       0-1       False       False     neither
  Denmark        Peru    1-0 1.95-0.40       1-0       2-0        True       False ✓ mode wins
Argentina     Iceland    1-1 0.98-0.70       0-0       1-0       False       False     neither
   Sweden South Korea    1-0 1.93-0.47       1-0       2-0        True       False ✓ mode wins
   Brazil  Costa Rica    2-0 2.92-0.63       2-0       3-0        True       False ✓ mode wins
  Morocco       Spain    2-2 0.94-1.60       0-1       1-1       False       False     neither
   Sweden      Mexico    3-0 1.96-0.36       1-0       2-0       False       False     neither

Lambda distribution (2018 WC, 128 slots):
  [0.00, 0.90):  63  ███████████████████████████████████████████████████████████████  mode=0, th90=0
  [0.90, 1.00):   3  ███  mode=0, th90=1  ← DIFFERS
  [1.00